In [ ]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker('pt_BR')

# 1. Dicionário base de estabelecimentos e categorias reais
BASE_MERCHANTS = {
    "Alimentacao": ["IFOOD", "RAPPI", "MCDONALDS", "BURGER KING", "RESTAURANTE DO JOAO", "SUPERMERCADO BRETAS", "CARREFOUR", "PADARIA SAO JOSE"],
    "Transporte": ["UBER", "99APP", "METRO SP", "SEMPARAR", "POSTO IPIRANGA", "SHELL BOX", "LOCALIZA"],
    "Assinaturas": ["NETFLIX", "SPOTIFY", "AMAZON PRIME", "DISNEY PLUS", "GLOBO PLAY"],
    "Educacao": ["UDEMY", "ALURA", "ESTACIO", "USP", "LIVRARIA CULTURA"],
    "Saude": ["DROGASIL", "RAIA", "CLINICA SAO LUCAS", "UNIMED", "DR CONSULTA"]
}

# 2. Motor de Ruído (Gateways e Padrões de Fatura)
PREFIXOS_GATEWAY = ["PAG*", "MP*", "MERCADOPAGO*", "PAYPAL *", "PGTO*", "IFD*", "APP*"]
CIDADES = ["SP", "SAO PAULO", "RIO DE JANEI", "BH", "CURITIBA"]

def aplicar_ruido_fatura(merchant):
    descricao = merchant.upper()
    
    # Simula 30% de chance de ter um prefixo de gateway de pagamento
    if random.random() < 0.3:
        prefixo = random.choice(PREFIXOS_GATEWAY)
        descricao = f"{prefixo}{descricao}"
        
    # Simula 20% de chance de ter a cidade no final (comum em maquininhas físicas)
    if random.random() < 0.2:
        cidade = random.choice(CIDADES)
        descricao = f"{descricao} {cidade}"
        
    # Simula erro de digitação ou remoção de espaços
    if random.random() < 0.1:
        descricao = descricao.replace(" ", "")
        
    # Regra de Ouro: Faturas de cartão geralmente truncam com 20-22 caracteres
    return descricao[:22]

# 3. Gerador do Dataset
def gerar_dataset_faturas(num_linhas=1000):
    dados = []
    
    for _ in range(num_linhas):
        categoria = random.choice(list(BASE_MERCHANTS.keys()))
        merchant_original = random.choice(BASE_MERCHANTS[categoria])
        
        # Gera a descrição como ela apareceria no aplicativo do cartão
        descricao_fatura = aplicar_ruido_fatura(merchant_original)
        
        # Gera valores coerentes (ex: assinaturas são mais baratas, educação mais cara)
        if categoria == "Assinaturas":
            valor = round(random.uniform(15.0, 60.0), 2)
        elif categoria == "Transporte":
            valor = round(random.uniform(8.0, 150.0), 2)
        else:
            valor = round(random.uniform(20.0, 500.0), 2)
            
        data_transacao = fake.date_between(start_date='-6m', end_date='today')
        
        dados.append({
            "Data": data_transacao,
            "Descricao_Fatura": descricao_fatura,
            "Valor": valor,
            "Categoria_Alvo": categoria # O que seu modelo terá que prever
        })
        
    return pd.DataFrame(dados)

# Executando a geração
df_faturas = gerar_dataset_faturas(5000)
df_faturas.to_csv("dataset_faturas_sintetico.csv", index=False)
print("Dataset gerado com sucesso!")

Dataset gerado com sucesso!


: 

In [ ]:
from gen_fraud_graph import Config, FraudGraphGenerator

config = Config(
    scale_factor=0.001,         # ~10K accounts, ~90K transactions
    num_fraud_rings=50,         # 50 cyclic fraud patterns
    embedding_provider="fake",  # random vectors (fast, no model needed)
    workers=2,                  # 2 parallel processes
    output_dir="./output",
)

generator = FraudGraphGenerator(config)
generator.run()